In [ ]:
!pip install feedparser

import feedparser
import pandas as pd
from datetime import datetime

print("Libraries imported successfully!")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 3.1 MB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=a759c5b9566c01504d9725637583650a7f2afa8d666edbcae5ee8f7b640d10e3
  Stored in directory: /root/.cache/pip/wheels/03/f5/1a/23761066dac1d0e8e683e5fdb27e12de53209d05a4a37e6246
Successfully built sgmllib3k
Libraries imported successfully!


In [ ]:

RSS_URL = "https://www.reddit.com/r/technology/.rss"

feed = feedparser.parse(RSS_URL)

raw_posts = []


for entry in feed.entries:
    raw_posts.append({
        "title": entry.title,
        "published": entry.updated,
        "link": entry.link
    })


df = pd.DataFrame(raw_posts)

df.head()

,title,published,link
0,AMD silently removes memory encryption from co...,2026-06-18T03:42:24+00:00,https://www.reddit.com/r/technology/comments/1...
1,"Firefox has an ambitious new roadmap, the brow...",2026-06-17T20:35:36+00:00,https://www.reddit.com/r/technology/comments/1...
2,Apple’s weird anti-nausea dots cured my car si...,2026-06-18T02:03:39+00:00,https://www.reddit.com/r/technology/comments/1...
3,"Experts warn ""colossal"" breach exposes 24 bill...",2026-06-17T23:47:00+00:00,https://www.reddit.com/r/technology/comments/1...
4,MIT Study Finds Gas Cars Aren't Secretly Bette...,2026-06-17T12:42:58+00:00,https://www.reddit.com/r/technology/comments/1...


In [ ]:

df['published'] = pd.to_datetime(df['published'])


df = df.sort_values(by='published', ascending=False).reset_index(drop=True)

print(f"Successfully collected {len(df)} live posts!")
df.head()

Successfully collected 25 live posts!


,title,published,link
0,Meta CTO reports employee morale is near histo...,2026-06-18 06:01:55+00:00,https://www.reddit.com/r/technology/comments/1...
1,AMD silently removes memory encryption from co...,2026-06-18 03:42:24+00:00,https://www.reddit.com/r/technology/comments/1...
2,US pulls the 'kill-switch' on Anthropic's Fabl...,2026-06-18 03:40:15+00:00,https://www.reddit.com/r/technology/comments/1...
3,Apple’s weird anti-nausea dots cured my car si...,2026-06-18 02:03:39+00:00,https://www.reddit.com/r/technology/comments/1...
4,"Tesco moving 40,000 server workloads off VMwar...",2026-06-18 01:18:23+00:00,https://www.reddit.com/r/technology/comments/1...


In [ ]:
import re
import nltk


nltk.download('stopwords')
from nltk.corpus import stopwords


STOP_WORDS = set(stopwords.words('english'))

print("Text processing resources downloaded!")

Text processing resources downloaded!


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
def clean_text(text):

    text = text.lower()

    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    text = re.sub(r'\[.*?\]', '', text)

    text = re.sub(r'[^a-zA-Z\s]', '', text)

    words = text.split()

    cleaned_words = [w.strip() for w in words if w.strip() not in STOP_WORDS]

    return " ".join(cleaned_words)

print("Cleaning function defined successfully!")

Cleaning function defined successfully!


In [ ]:
df['cleaned_title'] = df['title'].apply(clean_text)

df[['title', 'cleaned_title']].head()

,title,cleaned_title
0,Meta CTO reports employee morale is near histo...,meta cto reports employee morale near historic...
1,AMD silently removes memory encryption from co...,amd silently removes memory encryption consume...
2,US pulls the 'kill-switch' on Anthropic's Fabl...,us pulls killswitch anthropics fable ai models...
3,Apple’s weird anti-nausea dots cured my car si...,apples weird antinausea dots cured car sicknes...
4,"Tesco moving 40,000 server workloads off VMwar...",tesco moving server workloads vmware amid broa...


In [ ]:

nltk.download('vader_lexicon')
from nltk.sentiment.vader import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

print("VADER Sentiment Analyzer initialized successfully!")

VADER Sentiment Analyzer initialized successfully!


[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


In [ ]:
def get_sentiment_label(score):

    if score >= 0.05:
        return 'Positive'
    elif score <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

test_score = sia.polarity_scores("Artificial Intelligence is amazing but dangerous!")['compound']
print(f"Test Compound Score: {test_score} -> Label: {get_sentiment_label(test_score)}")

Test Compound Score: -0.2481 -> Label: Negative


In [ ]:
df['sentiment_score'] = df['title'].apply(lambda x: sia.polarity_scores(x)['compound'])

df['sentiment_category'] = df['sentiment_score'].apply(get_sentiment_label)

df[['title', 'sentiment_score', 'sentiment_category']].head(10)

,title,sentiment_score,sentiment_category
0,Meta CTO reports employee morale is near histo...,0.1531,Positive
1,AMD silently removes memory encryption from co...,-0.0772,Negative
2,US pulls the 'kill-switch' on Anthropic's Fabl...,-0.3400,Negative
3,Apple’s weird anti-nausea dots cured my car si...,0.3498,Positive
4,"Tesco moving 40,000 server workloads off VMwar...",0.0000,Neutral
5,SanDisk's new 8TB PS5 SSD costs more than thre...,0.0000,Neutral
6,"Experts warn ""colossal"" breach exposes 24 bill...",-0.2263,Negative
7,U.S. science is in chaos — Today the most infl...,0.2716,Positive
8,Allbirds continues AI pivot with name change a...,0.0000,Neutral
9,After unveiling ridiculously expensive AR glas...,-0.3400,Negative


In [ ]:
trend_summary = df['sentiment_category'].value_counts(normalize=True) * 100

print("--- Current Subreddit Sentiment Trend Distribution ---")
for category, percentage in trend_summary.items():
    print(f"{category}: {percentage:.1f}%")

--- Current Subreddit Sentiment Trend Distribution ---
Negative: 44.0%
Neutral: 36.0%
Positive: 20.0%


In [ ]:
dashboard_df = df[['published', 'title', 'sentiment_score', 'sentiment_category']].copy()

dashboard_df.columns = ['Timestamp', 'Raw_Title', 'Sentiment_Score', 'Sentiment_Class']

print("--- Prepared Frontend Data Layout ---")
dashboard_df.head()

--- Prepared Frontend Data Layout ---


,Timestamp,Raw_Title,Sentiment_Score,Sentiment_Class
0,2026-06-18 06:01:55+00:00,Meta CTO reports employee morale is near histo...,0.1531,Positive
1,2026-06-18 03:42:24+00:00,AMD silently removes memory encryption from co...,-0.0772,Negative
2,2026-06-18 03:40:15+00:00,US pulls the 'kill-switch' on Anthropic's Fabl...,-0.3400,Negative
3,2026-06-18 02:03:39+00:00,Apple’s weird anti-nausea dots cured my car si...,0.3498,Positive
4,2026-06-18 01:18:23+00:00,"Tesco moving 40,000 server workloads off VMwar...",0.0000,Neutral


In [ ]:
OUTPUT_FILE = "processed_sentiment_data.csv"
dashboard_df.to_csv(OUTPUT_FILE, index=False)

print(f"Success! Your data has been exported locally as '{OUTPUT_FILE}'.")

Success! Your data has been exported locally as 'processed_sentiment_data.csv'.
